# 🌙 PyDiff - Pyramid Diffusion Models for Low-light Image Enhancement

> **Referencia:** Zhou et al., *"Pyramid Diffusion Models For Low-light Image Enhancement"*, IJCAI 2023 (Oral)
> **Repositorio:** https://github.com/limuloo/PyDIff

---

## 📌 Visao Geral

Imagens capturadas em ambientes com pouca luz sofrem de:
- **Ruido intenso** que encobre detalhes finos
- **Cores distorcidas** e baixo contraste

O **PyDiff** resolve esses problemas combinando dois ingredientes:

| Componente | Papel |
|---|---|
| **Pyramid Diffusion** | Amostragem em resolucoes progressivamente crescentes -> muito mais rapido que diffusion vanilla |
| **Global Corrector** | Corrige degradacoes globais (e.g. RGB shift) que o denoiser ignora |

**Pipeline resumido:**
```
Imagem baixa luz (x_low)
        |
        ▼
[Processo reverso em piramide de resolucoes]
  T passos @ resolucao baixa  ->  upscale
  T passos @ resolucao media  ->  upscale
  T passos @ resolucao original
        |
        ▼
  [Global Corrector]
        |
        ▼
   Imagem aprimorada (x_0)
```

[!] **Pre-requisito:** Ative o runtime GPU em `Runtime -> Change runtime type -> T4 GPU`

## 1. Instalacao do Ambiente

O PyDiff e construido sobre o **BasicSR**, framework popular para restauracao de imagens.
A instalacao encadeia dois pacotes: `BasicSR-light` (backbone) e `PyDiff` (modelo).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
if [ ! -d "PyDIff" ]; then
    git clone https://github.com/limuloo/PyDIff.git
    echo "[OK] Repositorio clonado!"
else
    echo "[OK] Repositorio ja existe."
fi

In [ ]:
%%bash
cd PyDIff/BasicSR-light
pip install -q -r requirements.txt
BASICSR_EXT=True python setup.py develop --quiet 2>&1 | tail -3
echo "[OK] BasicSR-light instalado!" 

In [ ]:
%%bash
cd PyDIff/PyDiff
pip install -q -r requirements.txt
BASICSR_EXT=True python setup.py develop --quiet 2>&1 | tail -3
echo "[OK] PyDiff instalado!" 

## 2. Dataset LOL e Modelo Pre-treinado

### Dataset LOL (Low-Light)
Benchmark padrao - 485 pares treino + 15 pares teste (baixa luz / luz normal).

```
LOLdataset/
  our485/      ← treino: 485 pares
    low/       ← imagens com pouca luz
    high/      ← ground truth
  eval15/      ← teste: 15 pares
```

> **Download:** https://daooshee.github.io/BMVC2018website/
> **Pesos:** https://github.com/limuloo/PyDIff (secao *Pretrained Models*) -> `LOLweights.pth`

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path

DATASET_DIR  = "PyDIff/PyDiff/dataset/LOLdataset"
PRETRAINED   = "PyDIff/PyDiff/pretrained_models"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(PRETRAINED,  exist_ok=True)

# --- Gera imagens sinteticas para rodar o notebook sem o dataset completo ---
def create_synthetic_lowlight(output_dir, n=5, size=(400, 600)):
    '''Cria pares sinteticos (baixa luz / normal) para demonstracao.'''
    low_dir  = Path(output_dir) / "eval15" / "low"
    high_dir = Path(output_dir) / "eval15" / "high"
    low_dir.mkdir(parents=True, exist_ok=True)
    high_dir.mkdir(parents=True, exist_ok=True)

    np.random.seed(42)
    for i in range(n):
        h, w = size
        # cena "normal" com gradiente + formas coloridas
        high = np.zeros((h, w, 3), dtype=np.uint8)
        high[:,:,0] = np.tile(np.linspace(50, 220, w), (h, 1)).astype(np.uint8)
        high[:,:,1] = np.tile(np.linspace(100, 180, h)[:,None], (1, w)).astype(np.uint8)
        high[:,:,2] = 120
        for _ in range(8):
            cx, cy = np.random.randint(50, w-50), np.random.randint(50, h-50)
            color  = tuple(np.random.randint(100, 255, 3).tolist())
            cv2.circle(high, (cx, cy), np.random.randint(20, 60), color, -1)

        # "baixa luz" = escurece 85% + ruido gaussiano
        low   = (high * 0.15).astype(np.int16)
        noise = np.random.normal(0, 8, low.shape).astype(np.int16)
        low   = np.clip(low + noise, 0, 255).astype(np.uint8)

        cv2.imwrite(str(high_dir / f"{i:04d}.png"), cv2.cvtColor(high, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(low_dir  / f"{i:04d}.png"), cv2.cvtColor(low,  cv2.COLOR_RGB2BGR))

    return low_dir, high_dir

low_dir, high_dir = create_synthetic_lowlight(DATASET_DIR)
print(f"[OK] 5 pares sinteticos criados em {DATASET_DIR}")

## 3. Fundamentos Teoricos

### 3.1 Diffusion Models - Revisao Rapida

**Forward process** - adiciona ruido gaussiano em T passos:

$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\, \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\, \beta_t\mathbf{I}\right)$$

**Reverse process** - a rede remove o ruido condicionada em $\mathbf{x}_{low}$:

$$p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t, \mathbf{x}_{low}) = \mathcal{N}\!\left(\mathbf{x}_{t-1};\, \mu_\theta(\mathbf{x}_t, \mathbf{x}_{low}, t),\, \Sigma_\theta\right)$$

### 3.2 Pyramid Diffusion

O problema da diffusion vanilla e a **resolucao constante** em todos os T passos - caro.

A solucao: processar em **piramide de resolucoes crescentes**:

```
Passos 0   -> T1  |  resolucao R/4  (barato, captura estrutura global)
                  |  v upsample bilinear
Passos T1  -> T2  |  resolucao R/2  (detalhes medios)
                  |  v upsample bilinear
Passos T2  -> T   |  resolucao R    (detalhes finos)
```

### 3.3 Global Corrector

O denoiser e treinado para eliminar **ruido gaussiano de media zero**.
Desvios globais de cor (RGB shift) acumulados no processo reverso nao sao eliminados por ele.
O **Global Corrector** e uma UNet leve que recebe $(\mathbf{x}_0^{diff}, \mathbf{x}_{low})$ e produz a saida final.

## 4. Visualizando o Forward Diffusion Process

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

def load_img(path, size=(256, 384)):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, size)

def img_to_tensor(img):
    '''HWC uint8 para (1,C,H,W) float em [-1,1]'''
    t = torch.from_numpy(img).permute(2,0,1).float() / 127.5 - 1.0
    return t.unsqueeze(0)

def tensor_to_img(t):
    '''(1,C,H,W) float para HWC uint8'''
    t = t.squeeze(0).permute(1,2,0)
    return ((t + 1.0) * 127.5).clamp(0,255).byte().numpy()

# -- Cosine noise schedule -----------------------------------------
def cosine_beta_schedule(T=1000, s=0.008):
    steps = torch.arange(T+1, dtype=torch.float64)
    acp   = torch.cos(((steps/T) + s) / (1+s) * torch.pi/2) ** 2
    acp   = acp / acp[0]
    betas = 1 - (acp[1:] / acp[:-1])
    return betas.clamp(0, 0.999).float()

T           = 1000
betas       = cosine_beta_schedule(T)
alphas      = 1.0 - betas
alphas_cp   = torch.cumprod(alphas, 0)
sqrt_acp    = alphas_cp.sqrt()
sqrt_1m_acp = (1 - alphas_cp).sqrt()

def q_sample(x0, t, noise=None):
    '''Adiciona ruido em t passos de uma vez (reparametrizacao fechada).'''
    if noise is None:
        noise = torch.randn_like(x0)
    a = sqrt_acp[t].view(-1,1,1,1)
    b = sqrt_1m_acp[t].view(-1,1,1,1)
    return a * x0 + b * noise

# -- Visualiza -----------------------------------------------------
sample_high = sorted(Path(high_dir).glob("*.png"))[0]
sample_low  = sorted(Path(low_dir).glob("*.png"))[0]
img_np      = load_img(sample_high)
x0          = img_to_tensor(img_np)

timesteps   = [0, 100, 250, 500, 750, 999]
fig, axes   = plt.subplots(1, len(timesteps), figsize=(16, 3))
fig.suptitle("Forward Diffusion  q(x_t | x_0)", fontsize=13, weight='bold')

for ax, t_val in zip(axes, timesteps):
    xt  = q_sample(x0, torch.tensor([t_val]))
    snr = (sqrt_acp[t_val] / sqrt_1m_acp[t_val]).item()
    ax.imshow(tensor_to_img(xt))
    ax.set_title(f"t = {t_val}
SNR = {snr:.2f}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Piramide de Resolucoes - Visualizacao

In [ ]:
def build_pyramid(img, scales=[0.25, 0.5, 1.0]):
    h, w = img.shape[:2]
    return [(s, cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA))
            for s in scales]

img_high_np = load_img(sample_high, size=(256, 384))
img_low_np  = load_img(sample_low,  size=(256, 384))
scales      = [0.25, 0.5, 1.0]
pyr_h = build_pyramid(img_high_np, scales)
pyr_l = build_pyramid(img_low_np,  scales)

stage_names = ["Estagio 1
(1/4 da res.)", "Estagio 2
(1/2 da res.)", "Estagio 3
(resolucao original)"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for col in range(3):
    for row, pyr in enumerate([pyr_h, pyr_l]):
        _, img_lvl = pyr[col]
        axes[row, col].imshow(img_lvl)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f"{stage_names[col]}
{img_lvl.shape[1]}x{img_lvl.shape[0]} px", fontsize=9)

axes[0,0].set_ylabel("Ground Truth", fontsize=9)
axes[1,0].set_ylabel("Baixa Luz (x_low)", fontsize=9)

# setas entre colunas
for col in range(2):
    x = (col + 1)/3 - 0.01
    fig.text(x, 0.5, "->", fontsize=24, ha='center', va='center', color='steelblue', weight='bold')

fig.suptitle("PyDiff: denoising parte da menor resolucao e vai crescendo progressivamente",
             fontsize=11, weight='bold')
plt.tight_layout()
plt.show()

## 6. Configuracao YAML

O BasicSR usa arquivos `.yaml` para controlar todo o pipeline.

In [ ]:
import yaml, copy

infer_yaml = Path("PyDIff/PyDiff/options/infer.yaml")

if infer_yaml.exists():
    with open(infer_yaml) as f:
        config = yaml.safe_load(f)

    print("=" * 50)
    print("   Configuracao de Inferencia (infer.yaml)")
    print("=" * 50)
    for k in ['name', 'scale', 'network_g', 'val']:
        if k in config:
            print(f"
[{k}]")
            print(yaml.dump({k: config[k]}, default_flow_style=False, indent=2))

    # Personaliza caminhos para o Colab
    config_mod = copy.deepcopy(config)
    root = Path("PyDIff/PyDiff").resolve()
    for ds in config_mod.get('datasets', {}).values():
        if 'dataroot_lq' in ds: ds['dataroot_lq'] = str(root/'dataset/LOLdataset/eval15/low')
        if 'dataroot_gt' in ds: ds['dataroot_gt'] = str(root/'dataset/LOLdataset/eval15/high')
    if 'path' in config_mod:
        config_mod['path']['pretrain_network_g'] = str(root/'pretrained_models/LOLweights.pth')

    out = Path("PyDIff/PyDiff/options/infer_colab.yaml")
    with open(out, 'w') as f:
        yaml.dump(config_mod, f)
    print(f"
[OK] Config personalizada salva em: {out}")
else:
    print("[!]  infer.yaml nao encontrado - verifique se o repo foi clonado.")

## 7. Inferencia com o Modelo Pre-treinado

> **[!] Pre-requisito:** baixe `LOLweights.pth` e coloque em `PyDIff/PyDiff/pretrained_models/`
> Link: https://github.com/limuloo/PyDIff (secao *Pretrained Models*)

In [ ]:
weights = "PyDIff/PyDiff/pretrained_models/LOLweights.pth"
if Path(weights).exists():
    print(f"[OK] Pesos encontrados: {Path(weights).stat().st_size/1e6:.1f} MB")
else:
    print("[X] Pesos NAO encontrados.")
    print("   Baixe LOLweights.pth e coloque em:", weights)

In [ ]:
%%bash
WEIGHTS="PyDIff/PyDiff/pretrained_models/LOLweights.pth"
if [ -f "$WEIGHTS" ]; then
    cd PyDIff/PyDiff
    CUDA_VISIBLE_DEVICES=0 python pydiff/train.py -opt options/infer_colab.yaml
    echo "[OK] Inferencia concluida!"
else
    echo "[>>]  Pulando - pesos ausentes. Continue para a proxima secao."
fi

## 8. Metricas Quantitativas - PSNR e SSIM

- **PSNR** (Peak Signal-to-Noise Ratio): mede distorcao pixel a pixel. Quanto maior, melhor.
- **SSIM** (Structural Similarity): mede similaridade perceptual de estruturas. Varia de 0 a 1.

In [ ]:
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import pandas as pd

def compute_metrics(pred, gt):
    p = psnr_fn(gt, pred, data_range=255)
    s = ssim_fn(gt, pred, channel_axis=2, data_range=255)
    return p, s

low_paths  = sorted(Path(low_dir).glob("*.png"))
high_paths = sorted(Path(high_dir).glob("*.png"))
results_dir = Path("PyDIff/PyDiff/results")
enh_paths   = sorted(results_dir.glob("**/*.png")) if results_dir.exists() else []

records = []
for i, (lp, hp) in enumerate(zip(low_paths, high_paths)):
    img_l = load_img(lp)
    img_h = load_img(hp)
    p_in, s_in = compute_metrics(img_l, img_h)
    rec = {"Imagem": Path(hp).stem, "PSNR_input": round(p_in,2), "SSIM_input": round(s_in,4)}

    if enh_paths and i < len(enh_paths):
        img_e = load_img(enh_paths[i])
        p_en, s_en = compute_metrics(img_e, img_h)
        rec["PSNR_pydiff"] = round(p_en,2)
        rec["SSIM_pydiff"] = round(s_en,4)
        rec["DeltaPSNR"] = round(p_en - p_in, 2)
    records.append(rec)

df = pd.DataFrame(records)
print("[chart] Metricas por imagem:")
display(df)
print("
[up] Medias:")
display(df.select_dtypes('number').mean().round(3).to_frame("Media").T)
print("
[tip] Referencia PyDiff (LOL-v1 eval15): PSNR ~= 23.65 | SSIM ~= 0.843")

## 9. Comparacao Visual - Baixa Luz x Ground Truth x PyDiff

In [ ]:
def visualize_comparison(low_paths, high_paths, enh_paths=None, n=3):
    n_cols  = 3 if enh_paths else 2
    titles  = ["Entrada (baixa luz)", "Ground Truth"]
    if enh_paths: titles.append("PyDiff Output")

    fig, axes = plt.subplots(n, n_cols, figsize=(5*n_cols, 4*n))
    if n == 1: axes = axes[None]

    for row in range(n):
        img_l = load_img(low_paths[row])
        img_h = load_img(high_paths[row])
        p_in, s_in = compute_metrics(img_l, img_h)

        axes[row,0].imshow(img_l);  axes[row,0].axis('off')
        axes[row,0].set_xlabel(f"PSNR={p_in:.1f} SSIM={s_in:.3f}", fontsize=8, color='firebrick')
        axes[row,1].imshow(img_h);  axes[row,1].axis('off')
        axes[row,1].set_xlabel("Referencia", fontsize=8)

        if enh_paths and row < len(enh_paths):
            img_e = load_img(enh_paths[row])
            p_e, s_e = compute_metrics(img_e, img_h)
            axes[row,2].imshow(img_e); axes[row,2].axis('off')
            axes[row,2].set_xlabel(f"PSNR={p_e:.1f} SSIM={s_e:.3f}", fontsize=8, color='forestgreen')

    for col, t in enumerate(titles):
        axes[0,col].set_title(t, fontsize=10, weight='bold')

    plt.suptitle("Comparacao: Baixa Luz x Ground Truth x PyDiff", fontsize=12, weight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

visualize_comparison(low_paths[:3], high_paths[:3],
                     enh_paths[:3] if enh_paths else None)

## 10. Analise de Histograma - Distribuicao de Brilho

O PyDiff deve deslocar o histograma de luminancia da entrada (concentrado em valores baixos) para proximo do ground truth.

In [ ]:
def plot_lum_histogram(img_low, img_high, img_enh=None):
    def luminance(img):
        return cv2.cvtColor(img, cv2.COLOR_RGB2HSV)[:,:,2].flatten()

    bins = np.arange(0, 257, 4)
    fig, ax = plt.subplots(figsize=(9, 4))

    ax.hist(luminance(img_low),  bins=bins, alpha=0.6, color='royalblue', label='Baixa Luz')
    ax.hist(luminance(img_high), bins=bins, alpha=0.6, color='gold',      label='Ground Truth')
    if img_enh is not None:
        ax.hist(luminance(img_enh), bins=bins, alpha=0.6, color='limegreen', label='PyDiff')

    ax.set_xlabel("Luminancia (canal V do HSV)", fontsize=11)
    ax.set_ylabel("Contagem de pixels",          fontsize=11)
    ax.set_title("Distribuicao de Brilho - PyDiff aproxima o histograma do Ground Truth",
                 fontsize=11, weight='bold')
    ax.legend(fontsize=10); ax.set_xlim(0, 255)
    plt.tight_layout(); plt.show()

img_enh_sample = load_img(enh_paths[0]) if enh_paths else None
plot_lum_histogram(load_img(low_paths[0]), load_img(high_paths[0]), img_enh_sample)

## 11. Simulacao do Reverse Process Piramidal

Demonstracao conceitual da estrutura do algoritmo - sem o denoiser real.

In [ ]:
class PyramidDiffSimulator:
    '''
    Simula o processo reverso piramidal.
    Em vez da UNet real, usa x_low como pseudo-predicao de x0
    para mostrar a estrutura do algoritmo.
    '''
    def __init__(self, betas, levels=[(0.25, 300), (0.5, 300), (1.0, 400)]):
        self.alphas   = 1.0 - betas
        self.acp      = torch.cumprod(self.alphas, 0)
        self.betas    = betas
        self.levels   = levels

    def ddpm_step(self, xt, x0_pred, t):
        beta   = self.betas[t]
        acp    = self.acp[t]
        acp_pm = self.acp[t-1] if t > 0 else torch.tensor(1.0)
        c1  = beta * acp_pm.sqrt() / (1 - acp)
        c2  = (1 - acp_pm) * self.alphas[t].sqrt() / (1 - acp)
        mu  = c1 * x0_pred + c2 * xt
        var = beta * (1 - acp_pm) / (1 - acp)
        z   = torch.randn_like(xt) if t > 0 else torch.zeros_like(xt)
        return mu + var.sqrt() * z

    @torch.no_grad()
    def run(self, x_low_np, n_snaps=6):
        h, w    = x_low_np.shape[:2]
        x_low   = img_to_tensor(x_low_np)
        T_total = sum(s[1] for s in self.levels)
        t_cur   = T_total - 1

        s0, _   = self.levels[0]
        xt      = torch.randn(1, 3, int(h*s0), int(w*s0))
        snaps, labels = [], []

        for lvl, (scale, n_steps) in enumerate(self.levels):
            hl, wl = int(h*scale), int(w*scale)
            xl = torch.nn.functional.interpolate(x_low, (hl, wl), mode='bilinear', align_corners=False)
            if xt.shape[-2] != hl:
                xt = torch.nn.functional.interpolate(xt, (hl, wl), mode='bilinear', align_corners=False)

            ts_level = list(range(t_cur, t_cur - n_steps, -1))
            snap_ts  = set(ts_level[::max(1, n_steps // max(1, n_snaps // len(self.levels)))])

            for t_val in ts_level:
                idx     = min(t_val, len(self.betas)-1)
                nl      = self.acp[idx].item()           # pseudo-predicao com x_low
                x0_pred = nl * xl + (1-nl) * xt.clamp(-1,1)
                xt      = self.ddpm_step(xt, x0_pred, idx)

                if t_val in snap_ts:
                    xt_up = torch.nn.functional.interpolate(xt, (h, w), mode='bilinear', align_corners=False)
                    snaps.append(tensor_to_img(xt_up))
                    labels.append(f"Nivel {lvl+1} ({int(scale*100)}%)
t={t_val}")

            t_cur -= n_steps
        return snaps, labels

sim = PyramidDiffSimulator(betas)
snaps, labels = sim.run(load_img(low_paths[0]), n_snaps=6)

fig, axes = plt.subplots(1, len(snaps), figsize=(18, 3.5))
fig.suptitle("Simulacao do Reverse Process Piramidal (sem denoiser real)", fontsize=12, weight='bold')
for ax, s, l in zip(axes, snaps, labels):
    ax.imshow(s); ax.set_title(l, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()
print("Nota: versao didatica - o modelo real usa a UNet treinada como denoiser.")

## 12. Global Corrector - Por que e necessario?

O denoiser e treinado para remover **ruido gaussiano de media zero**.
Ele nao corrige **desvios globais de cor** (RGB shift), pois esses nao sao ruido gaussiano.
O Global Corrector e uma UNet leve que recebe o output do diffusion + x_low e corrige esses desvios.

In [ ]:
def simulate_rgb_shift(img, shift=(40, -30, 20)):
    shifted = img.astype(np.int32)
    for ch, s in enumerate(shift):
        shifted[:,:,ch] += s
    return np.clip(shifted, 0, 255).astype(np.uint8)

def global_corrector_demo(img_deg, img_gt):
    '''Estima e corrige bias global por canal (simplificacao educacional).'''
    bias = img_gt.astype(np.float32).mean((0,1)) - img_deg.astype(np.float32).mean((0,1))
    corrected = np.clip(img_deg.astype(np.float32) + bias, 0, 255).astype(np.uint8)
    return corrected, bias

img_ref   = load_img(high_paths[0])
img_shift = simulate_rgb_shift(img_ref)
img_corr, bias = global_corrector_demo(img_shift, img_ref)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
imgs   = [img_shift, img_ref, img_corr]
titles = [
    "Output do diffusion
c/ RGB shift (Delta R=+40 G=-30 B=+20)",
    "Ground Truth",
    f"Apos Global Corrector
(bias: R={bias[0]:.1f} G={bias[1]:.1f} B={bias[2]:.1f})"
]
colors = ['firebrick', 'gray', 'forestgreen']
for ax, img, title, c in zip(axes, imgs, titles, colors):
    ax.imshow(img); ax.set_title(title, fontsize=9, color=c); ax.axis('off')
    p, s = compute_metrics(img, img_ref)
    ax.set_xlabel(f"PSNR={p:.1f}  SSIM={s:.3f}", fontsize=8)

plt.suptitle("Global Corrector: corrige desvios sistematicos de cor pos-diffusion",
             fontsize=11, weight='bold')
plt.tight_layout(); plt.show()

## 13. Arquitetura - Visao Geral

O denoiser do PyDiff e uma **UNet condicional** com attention, similar ao DDPM original.
Abaixo tentamos importar o modelo real; se nao disponivel, mostramos a estrutura resumida.

In [ ]:
import sys
sys.path.insert(0, "PyDIff/BasicSR-light")
sys.path.insert(0, "PyDIff/PyDiff")

try:
    from pydiff.models.archs import PyDiffArch
    model = PyDiffArch(in_channels=6, out_channels=3, model_channels=64,
                       num_res_blocks=2, attention_resolutions=[8,16],
                       channel_mult=[1,2,4,8])
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"[OK] Modelo carregado - {n_params:.1f}M parametros totais")
    print(model)
except Exception as e:
    print(f"[!]  Import nao disponivel: {e}
")
    print('''
    print('''
    PyDiff -- UNet Condicional
    --------------------------------------------------
    Input:  [x_t (3ch) concat x_low (3ch)] = 6 canais

    ENCODER
      ResBlock(ch=64)  x2  + time_embedding
      Downsample 2x
      ResBlock(ch=128) x2  + SelfAttention
      Downsample 2x
      ResBlock(ch=256) x2  + SelfAttention
      Downsample 2x
      ResBlock(ch=512) x2  + SelfAttention

    BOTTLENECK
      ResBlock(ch=512) + SelfAttention

    DECODER  (skip connections do encoder)
      ResBlock(ch=512) + Upsample
      ResBlock(ch=256) + Upsample
      ResBlock(ch=128) + Upsample
      ResBlock(ch=64)

    Output: Conv 3x3 -> 3 canais  (predicao do ruido epsilon)
    Time Embedding: sinusoidal -> Linear -> SiLU -> Linear
    Global Corrector: UNet leve (~1M params)
    --------------------------------------------------
    ''')

## 14. Treinamento - Hiperparametros e Comandos

> [*] Treinamento do zero requer GPU com >= 24GB VRAM e o dataset LOL completo.

In [ ]:
train_yaml = Path("PyDIff/PyDiff/options/train.yaml")

if train_yaml.exists():
    with open(train_yaml) as f:
        tc = yaml.safe_load(f)
    for k in ['train', 'network_g']:
        if k in tc:
            print(f"
[{k}]")
            print(yaml.dump({k: tc[k]}, default_flow_style=False, indent=2))
else:
    print('''Hiperparametros principais (paper IJCAI 2023):

  Otimizador:       Adam (lr = 1e-4)
  Batch size:       8
  Iteracoes:        300 000
  T (timesteps):    1000
  Schedule:         linear (beta_1=1e-4, beta_T=0.02)
  Loss:             L1 no espaco de ruido (epsilon-prediction)
  Crop de treino:   96 x 96 px
  kind_align:       True no val (alinha iluminacao global)

  Piramide de resolucoes:
    Nivel 1: escala 1/4  - passos 0 -> 333
    Nivel 2: escala 1/2  - passos 334 -> 666
    Nivel 3: escala 1/1  - passos 667 -> 999

  Global Corrector: treinado com loss separado apos diffusion.
''')

print("Comando para iniciar treino:")
print("  cd PyDIff/PyDiff")
print("  CUDA_VISIBLE_DEVICES=0 python pydiff/train.py -opt options/train.yaml")

## 15. Resumo e Proximos Passos

### O que aprendemos

| Conceito | PyDiff |
|---|---|
| **Diffusion condicional** | $\mathbf{x}_{low}$ concatenado com $\mathbf{x}_t$ guia o denoiser |
| **Piramide de resolucoes** | Denoising em 1/4 -> 1/2 -> 1/1 da resolucao original |
| **Global Corrector** | Rede leve que corrige RGB shift apos o processo reverso |
| **Resultado SOTA** | PSNR ~= 23.65, SSIM ~= 0.843 no LOL-v1 (IJCAI 2023 Oral) |

### Proximos passos

1. **Baixe `LOLweights.pth`** e rode a inferencia real na Secao 7
2. **Suas proprias imagens** - ajuste `use_kind_align: false` no yaml para imagens fora do dominio LOL
3. **Explorar LOL-v2** (synthetic / real) para avaliar generalizacao
4. **Comparar baselines** - EnlightenGAN, ZeroDCE, LLFLOW nas mesmas imagens
5. **Fine-tuning** - adapte para seu dominio (cameras de seguranca, medico, automotivo)

### Referencias

- Zhou et al. (2023) - [Pyramid Diffusion Models For Low-light Image Enhancement](https://arxiv.org/abs/2305.10028)
- Ho et al. (2020) - [Denoising Diffusion Probabilistic Models (DDPM)](https://arxiv.org/abs/2006.11239)
- Wang et al. (2022) - [LLFLOW](https://arxiv.org/abs/2110.09728)
- Dataset LOL - [BMVC 2018](https://daooshee.github.io/BMVC2018website/)